In [ ]:
# Cell 1: imports & config
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import pearsonr
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor 
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported")

In [ ]:
# Cell 1: Check that files exist

# Configure paths
CONFIG = {
    "model_params_path": "results/MOR/LGBM_MOR_best_params.json",
    "data_paths": {
        "X_train": "data/X_train.csv", 
        "y_train": "data/Y_train.csv"
    }
}

def check_files_exist():
    """Check that all required files exist"""
    print("=" * 60)
    print("File existence check")
    print("=" * 60)
    
    files_to_check = [
        ("Model parameters", CONFIG['model_params_path']),
        ("Feature data", CONFIG['data_paths']['X_train']),
        ("Target variables", CONFIG['data_paths']['y_train'])
    ]
    
    all_exist = True
    for file_desc, file_path in files_to_check:
        if os.path.exists(file_path):
            print(f"✓ {file_desc}: {file_path} - exists")
            # Show file size
            file_size = os.path.getsize(file_path)
            print(f"  File size: {file_size} bytes")
        else:
            print(f"✗ {file_desc}: {file_path} - missing")
            all_exist = False
    
    return all_exist

# Check file existence
files_exist = check_files_exist()

In [ ]:
# Cell 2: Configuration and data loading
# Configure paths
CONFIG = {
    "model_params_path": "results/MOR/LGBM_MOR_best_params.json",
    "data_paths": {
        "X_train": "data/X_train.csv", 
        "y_train": "data/Y_train.csv"
    },
    "output": {
        "residuals": "results/residuals/LGBM_MOR_train_residuals.csv",
        "partial_correlation": "results/correlation/incident_type_partial_correlations_regression.csv",
        "correlation_plot": "results/correlation/partial_correlation_heatmap_regression.png"
    }
}

def clean_feature_names_strict(df):
    """Strictly clean feature names, keeping only letters, digits and underscores"""
    cleaned_df = df.copy()
    new_columns = []
    for col in cleaned_df.columns:
        # Keep only letters, digits and underscores; replace other characters with underscores
        col_clean = re.sub(r'[^a-zA-Z0-9_]', '_', str(col))
        # Collapse repeated underscores
        col_clean = re.sub(r'_+', '_', col_clean)
        # Strip leading and trailing underscores
        col_clean = col_clean.strip('_')
        # Use a default name if empty
        if not col_clean:
            col_clean = 'feature'
        new_columns.append(col_clean)
    
    # Ensure column names are unique
    unique_columns = []
    seen = {}
    for col in new_columns:
        if col in seen:
            seen[col] += 1
            unique_columns.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            unique_columns.append(col)
    
    cleaned_df.columns = unique_columns
    return cleaned_df

# Load data
def load_data():
    """Load training data and strictly clean feature names"""
    X_train = pd.read_csv(CONFIG['data_paths']['X_train'])
    y_train = pd.read_csv(CONFIG['data_paths']['y_train'])
    
    print("Example raw feature names:")
    print(list(X_train.columns)[:10])
    
    # Strictly clean feature names
    X_train = clean_feature_names_strict(X_train)
    
    print(f"Training data shape: X_train {X_train.shape}, y_train {y_train.shape}")
    print(f"Targets: {list(y_train.columns)}")
    print(f"Example cleaned feature names:")
    print(list(X_train.columns)[:10])
    
    return X_train, y_train

X_train, y_train = load_data()

In [ ]:
# Cell 3: Load model parameters and initialize model
def load_model_params():
    """Load regression model parameters"""
    with open(CONFIG['model_params_path'], 'r') as f:
        model_params = json.load(f)
    return model_params

def initialize_model(params):
    """Initialize multi-output regressor"""
    base_model = LGBMRegressor(
        verbosity=-1,
        random_state=42
    )
    base_model.set_params(**params)
    return MultiOutputRegressor(base_model)

model_params = load_model_params()
model = initialize_model(model_params)

print("Regression model initialized")
print(f"Model parameters: {model_params}")

In [ ]:
# Cell 4: Train model
print("Training model...")
model.fit(X_train, y_train)
print("Model training finished!")

In [ ]:
# Cell 5: Compute training-set residuals (regression)
def calculate_residuals(model, X_train, y_train):
    """Compute residuals on the training set (actual - predicted)"""
    # Get predictions
    y_pred = model.predict(X_train)
    
    # Create residual DataFrame
    residuals = pd.DataFrame()
    
    for i, target_name in enumerate(y_train.columns):
        # Get predictions for target i
        pred_values = y_pred[:, i]
        
        # Compute residual: actual - predicted
        residuals[target_name] = y_train[target_name] - pred_values
    
    return residuals, y_pred

residuals, y_pred = calculate_residuals(model, X_train, y_train)

print("Residual computation finished")
print("Residual summary:")
print(residuals.describe())

# Save residuals
residuals.to_csv(CONFIG['output']['residuals'], index=False)
print(f"Residuals saved to: {CONFIG['output']['residuals']}")

In [ ]:
## Cell 7: Compute partial correlations (from regression residuals)
def calculate_partial_correlations(residuals):
    """
    Compute partial correlations between incident types from regression residuals
    Correlation of residuals reflects the conditional dependence between incident types, given the features
    """
    # Compute the correlation matrix of residuals
    partial_corr_matrix = residuals.corr()
    
    # Extract partial correlations for all variable pairs
    partial_corr_pairs = []
    incident_types = residuals.columns
    
    for i in range(len(incident_types)):
        for j in range(i+1, len(incident_types)):
            type1 = incident_types[i]
            type2 = incident_types[j]
            corr_value = partial_corr_matrix.iloc[i, j]
            
            # Compute p-value
            corr, p_value = pearsonr(residuals[type1], residuals[type2])
            
            partial_corr_pairs.append({
                'incident_type_1': type1,
                'incident_type_2': type2,
                'partial_correlation': corr_value,
                'abs_partial_correlation': abs(corr_value),
                'p_value': p_value,
                'significant': p_value < 0.05
            })
    
    # Convert to DataFrame and sort
    partial_corr_df = pd.DataFrame(partial_corr_pairs)
    partial_corr_df = partial_corr_df.sort_values('abs_partial_correlation', ascending=False)
    
    return partial_corr_matrix, partial_corr_df

partial_corr_matrix, partial_corr_df = calculate_partial_correlations(residuals)

print("Partial correlation computation finished")
print(f"Partial correlation matrix shape: {partial_corr_matrix.shape}")

In [ ]:
# Cell 10: Conditional independence analysis (regression)
def analyze_conditional_independence(partial_corr_df, independence_threshold=0.1):
    """
    Analyze conditional independence
    Partial correlations of regression residuals near 0 indicate that incident types are conditionally independent given the features
    """
    print("=" * 70)
    print("Conditional independence results (regression residuals)")
    print("=" * 70)
    
    total_pairs = len(partial_corr_df)
    significant_pairs = partial_corr_df[partial_corr_df['significant']]
    
    # Count pairs at each correlation level
    weak_corr = partial_corr_df[partial_corr_df['abs_partial_correlation'] < independence_threshold]
    moderate_corr = partial_corr_df[(partial_corr_df['abs_partial_correlation'] >= independence_threshold) & 
                                   (partial_corr_df['abs_partial_correlation'] < 0.3)]
    strong_corr = partial_corr_df[partial_corr_df['abs_partial_correlation'] >= 0.3]
    
    print(f"Total incident-type pairs: {total_pairs}")
    print(f"Significantly correlated pairs (p < 0.05): {len(significant_pairs)}")
    
    print(f"\nCorrelation distribution:")
    print(f"Weak (|ρ| < {independence_threshold}): {len(weak_corr)} pairs ({len(weak_corr)/total_pairs*100:.1f}%)")
    print(f"Moderate (0.1 ≤ |ρ| < 0.3): {len(moderate_corr)} pairs ({len(moderate_corr)/total_pairs*100:.1f}%)")
    print(f"Strong (|ρ| ≥ 0.3): {len(strong_corr)} pairs ({len(strong_corr)/total_pairs*100:.1f}%)")
    
    print(f"\nOverall statistics:")
    print(f"Mean absolute partial correlation: {partial_corr_df['abs_partial_correlation'].mean():.4f}")
    print(f"Median partial correlation: {partial_corr_df['partial_correlation'].abs().median():.4f}")
    print(f"Std of partial correlation: {partial_corr_df['partial_correlation'].std():.4f}")
    
    # Show the strongest correlated pairs
    if len(strong_corr) > 0:
        print(f"\nTop 5 strongest conditional dependencies:")
        for _, row in strong_corr.head().iterrows():
            sig_flag = "**" if row['significant'] else ""
            print(f"  {row['incident_type_1']} - {row['incident_type_2']}: {row['partial_correlation']:.3f}{sig_flag}")
    
    # Conditional independence conclusion
    weak_percentage = len(weak_corr) / total_pairs * 100
    print(f"\nConditional independence conclusion:")
    if weak_percentage > 90:
        print("✓ Strong evidence for conditional independence: over 90% of incident-type pairs are conditionally independent given the features")
    elif weak_percentage > 80:
        print("○ Moderate evidence for conditional independence: 80-90% of incident-type pairs are conditionally independent given the features") 
    elif weak_percentage > 70:
        print("△ Weak evidence for conditional independence: 70-80% of incident-type pairs are conditionally independent given the features")
    else:
        print("✗ No evidence for conditional independence: fewer than 70% of incident-type pairs are conditionally independent given the features")
    
    return weak_corr, moderate_corr, strong_corr

weak_corr, moderate_corr, strong_corr = analyze_conditional_independence(partial_corr_df)